# ML-04 — Search Intelligence Data Contract

**Lane 2: Refresh / Content Opportunity Scoring**, continuing Weeks 1–2. This notebook ranks pages for an editor's review using a **future traffic-decline proxy**. Decline is not evidence that refreshing a page will help. The independent editorial target proposed in Week 2 is still uncollected.

Run all cells in order with Python 3. Local setup: `python -m pip install pandas duckdb huggingface_hub scikit-learn nbformat nbclient ipykernel`. In Colab use the same command prefixed with `%pip` once. Authenticate using a local Hugging Face login, `HF_TOKEN` in the environment, or the Colab Secret named `HF_TOKEN`; never put a token in a cell. Accept the warehouse access terms first.

Sources: [contract skill](../../skills/writing-data-contracts/SKILL.md), [FlyRank data skill](../../skills/flyrank/flyrank-data/SKILL.md), [dictionary](../../docs/data-dictionary.md), and [warehouse](https://huggingface.co/datasets/FlyRank/internship-warehouse). Only March is read; June stays sealed. Cached Parquet files remain untracked. No starter CSV substitutes for warehouse evidence.

## 1. Unit of analysis + time window

The five contract answers:

1. **One row:** The source grain is one `report_date × client_hash_id × content_hash_id` (a page-day). The feature frame has one client–page at a single decision date, **2026-03-17**. IDs are grouping keys, never predictors.
2. **Tables:** Only `fact_content_daily_performance/month=2026-03/data_0.parquet` from the pinned warehouse revision below. No joins are required. Explicit per-page coverage checks replace assumptions that all clients have equal history. We do not need the query table or mutable snapshot content metadata for these five features.
3. **Window:** Features and reference exposure cover **March 1–14 inclusive**. March 15–16 are a two-day reporting buffer and are unused. At the start of March 17, rank pages for review. The observed outcome covers **March 17–30 inclusive**. Both comparison windows contain 14 days; March 31 is unused. The buffer is an assumption, not proof of historic publication time.
4. **Target/proxy:** `declined = 1` when outcome-window impressions are less than 80% of feature-window impressions; otherwise 0. Require all 14 valid GSC page-days in each window, and at least **100 feature-window impressions**. This new 14-day exposure floor replaces Week 1's 500-per-90-day policy; it is a declared training-slice choice, not an equivalent rescaling. At inference, eligibility depends only on past coverage/exposure; future coverage is required retrospectively to observe the label. Pages without future coverage have unknown outcomes, not negative labels.
5. **Deliberate exclusion:** Exclude `fact_content_query_90d`: its fixed final-window context can overlap the future relative to this March decision. Also exclude June and the `_sample` table, which is the final month rather than a random sample.

**Output/action:** A decline-risk ranking and reasons help an editor choose 20 pages to inspect for outdated material, missing coverage, or intent mismatch. The editor may refresh, investigate technical causes, or monitor. False positives waste review slots; false negatives delay attention. This is a retrospective decision-support exercise, not a refresh-impact estimate.

Section 3 prints exactly three verification SQL queries and their real outputs. A separate feature-construction query aggregates the verified slice; it is not another verification query.

In [1]:
from pathlib import Path
import os
import platform
import importlib.metadata
import pandas as pd
import numpy as np
import duckdb
from huggingface_hub import HfApi, hf_hub_download, get_token
from IPython.display import display, Markdown

SEED = 42
K = 20
REPO_ID = "FlyRank/internship-warehouse"
REVISION = "50cbf7c3909d07be4d1b5906b4d09e882e5acbf2"
PARTITION = "fact_content_daily_performance/month=2026-03/data_0.parquet"
MIN_PAST_IMPRESSIONS = 100
FEATURES = ["mean_daily_impressions", "ctr_pct", "impression_weighted_position",
            "active_day_share", "daily_impression_cv"]
assert len(FEATURES) == 5

# Find the repository from either its root or work/notebooks; Colab uses /content.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "skills/README.md").is_file()), Path.cwd())
CACHE = ROOT / "work/outputs/hf_cache"
CACHE.mkdir(parents=True, exist_ok=True)
print("Pinned warehouse revision:", REVISION)
print("Partition:", PARTITION)
print("Python:", platform.python_version())
print({p: importlib.metadata.version(p) for p in ["pandas", "duckdb", "huggingface_hub", "scikit-learn"]})

Pinned warehouse revision: 50cbf7c3909d07be4d1b5906b4d09e882e5acbf2
Partition: fact_content_daily_performance/month=2026-03/data_0.parquet
Python: 3.14.6
{'pandas': '3.0.6', 'duckdb': '1.5.5', 'huggingface_hub': '2.0.0', 'scikit-learn': '1.9.1'}


/var/home/tanzimul/Repos/github.com/tanzimul3islam/flyrank-ml-internship-starter/work/.venv/lib64/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Fields: feature / label / context / excluded

Every field read or derived has one role. The raw past-impression sum is a feature ingredient; the target also uses it as a reference denominator. Knowing the reference level does not reveal the later numerator.

| Role | Fields | Meaning / handling |
|---|---|---|
| Feature ingredients | Past `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`; derived `past_impressions`, `past_clicks`, `weighted_position_sum`, `position_impressions`, `past_active_days`, `past_impression_sd` | Read only March 1–14; generate exactly the five inputs below. |
| Label / proxy | Later `gsc_impressions`, `future_impressions`, `declined`; temporary `deliberate_label_copy` | Later information never enters the retained model. The copy exists only during the marked leak experiment. |
| Context | `report_date`, `client_hash_id`, `content_hash_id`, `gsc_data_available`, `ga4_data_available`, `source_rows`, `past_valid_days`, `future_valid_days`, `past_first`, `past_last`, `future_first`, `future_last`, `decision_date` | Windows, coverage, joins, splitting and availability audit; never predictors. `ga4_data_available` is audited but GA4 is not required for this GSC-only lane. |
| Excluded | Query-window data, June / `_sample`, GA4 measurements, snapshot content metadata, all other source columns | Avoid window overlap, final-month reuse, unnecessary tracking exclusions and unproven historic metadata. |

**Exactly five features and their availability:**

| Feature | Definition | Available when? |
|---|---|---|
| `mean_daily_impressions` | Past impressions / 14 | Knowable at the decision moment because it uses only March 1–14 exposure, assuming those reports arrived by March 17. |
| `ctr_pct` | 100 × past clicks / past impressions | Knowable at the decision moment because both counts come from March 1–14; 1.0 means 1%, not 100%. |
| `impression_weighted_position` | Σ(valid daily position × daily impressions) / Σ(impressions on valid-position days) | Knowable at the decision moment because only March 1–14 positions are used; nonpositive or missing positions are unknown. |
| `active_day_share` | Past days with positive impressions / 14 | Knowable at the decision moment because all 14 past days have confirmed GSC coverage, so measured zero is meaningful. |
| `daily_impression_cv` | Population SD of daily impressions / daily mean | Knowable at the decision moment because its variation uses only March 1–14; the exposure floor keeps the denominator positive. |

**Missingness:** `IS TRUE` excludes both FALSE and NULL availability flags. Valid GSC days additionally require nonmissing, nonnegative impressions and clicks. No unavailable row is treated as zero traffic. Position can remain missing; median imputation is fitted on training clients only, with no added indicator (five inputs maximum). The third verification query shows missingness separately by GA4-availability category; later summaries report feature missingness. All-empty training features stop execution rather than silently disappearing.

In [2]:
# Retrieve credentials without printing, embedding, or saving them in notebook outputs.
token = os.environ.get("HF_TOKEN") or get_token()
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None

# A pinned local cache is reusable without re-downloading or contacting the service.
try:
    source_path = hf_hub_download(REPO_ID, PARTITION, repo_type="dataset",
                                  revision=REVISION, cache_dir=CACHE, local_files_only=True)
except Exception:
    if not token:
        raise RuntimeError("Warehouse access is required: accept the gate and use hf auth login, "
                           "an HF_TOKEN environment variable, or a Colab HF_TOKEN Secret. "
                           "Do not paste a token into this notebook or chat.") from None
    try:
        source_path = hf_hub_download(REPO_ID, PARTITION, repo_type="dataset",
                                      revision=REVISION, cache_dir=CACHE, token=token)
    except Exception:
        raise RuntimeError("Could not read the gated March partition. Check access approval, "
                           "READ-token permission, and network connectivity; then rerun.") from None
finally:
    token = None

con = duckdb.connect()
con.execute("SET threads = 2")
con.execute("SET memory_limit = '2GB'")
# Relational projection prevents accidental reading/display of private or unused fields.
SOURCE_FIELDS = ["report_date", "client_hash_id", "content_hash_id", "gsc_data_available",
                 "ga4_data_available", "gsc_impressions", "gsc_clicks", "gsc_avg_position"]
relation = con.read_parquet(source_path)
missing = set(SOURCE_FIELDS) - set(relation.columns)
assert not missing, f"Warehouse schema changed; missing expected fields: {sorted(missing)}"
relation.project(", ".join(SOURCE_FIELDS)).create_view("march")
print("March partition loaded. Source fields verified; no final-month table was opened.")

March partition loaded. Source fields verified; no final-month table was opened.


## 3. Verify it with queries, then build five features and spring the trap

### Verification query 1 of 3 — grain

Run on the entire March partition before filtering. An empty result means no duplicated page-day keys; null keys are included as violations. Any violation stops feature construction.

In [3]:
GRAIN_SQL = """
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS rows_per_key
FROM march
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1 OR report_date IS NULL
    OR client_hash_id IS NULL OR content_hash_id IS NULL
LIMIT 5
"""
print(GRAIN_SQL)
grain_check = con.sql(GRAIN_SQL).df()
display(grain_check)
assert grain_check.empty, "Page-day grain failed; fix duplicates/keys before aggregation."
print("PASS: one row per date × client × page in this March partition.")


SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS rows_per_key
FROM march
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1 OR report_date IS NULL
    OR client_hash_id IS NULL OR content_hash_id IS NULL
LIMIT 5



,report_date,client_hash_id,content_hash_id,rows_per_key


PASS: one row per date × client × page in this March partition.


### Verification query 2 of 3 — row count and date span

Report both the physical partition and the GSC-available lane slice. The per-client span bounds reveal unequal histories without printing client identifiers. Counts are measured here rather than copied from whole-release documentation.

In [4]:
COUNT_SQL = """
WITH scopes AS (
    SELECT 'March partition' AS scope, * FROM march
    UNION ALL
    SELECT 'GSC-available lane' AS scope, * FROM march
    WHERE gsc_data_available IS TRUE
), per_client AS (
    SELECT scope, client_hash_id, MIN(report_date) AS first_date,
           MAX(report_date) AS last_date, COUNT(DISTINCT report_date) AS observed_dates
    FROM scopes GROUP BY scope, client_hash_id
), spans AS (
    SELECT scope, MIN(first_date) AS earliest_client_start,
           MAX(first_date) AS latest_client_start,
           MIN(last_date) AS earliest_client_end,
           MIN(observed_dates) AS min_dates_per_client,
           MAX(observed_dates) AS max_dates_per_client
    FROM per_client GROUP BY scope
)
SELECT s.scope, COUNT(*) AS rows, MIN(s.report_date) AS first_date,
       MAX(s.report_date) AS last_date, COUNT(DISTINCT s.report_date) AS dates,
       COUNT(DISTINCT s.client_hash_id) AS clients,
       COUNT(DISTINCT (s.client_hash_id, s.content_hash_id)) AS pages,
       ANY_VALUE(p.earliest_client_start) AS earliest_client_start,
       ANY_VALUE(p.latest_client_start) AS latest_client_start,
       ANY_VALUE(p.earliest_client_end) AS earliest_client_end,
       ANY_VALUE(p.min_dates_per_client) AS min_dates_per_client,
       ANY_VALUE(p.max_dates_per_client) AS max_dates_per_client
FROM scopes s JOIN spans p USING (scope)
GROUP BY s.scope ORDER BY s.scope
"""
print(COUNT_SQL)
counts = con.sql(COUNT_SQL).df()
display(counts)
assert len(counts) == 2 and counts["rows"].gt(0).all()
assert pd.to_datetime(counts["first_date"]).ge(pd.Timestamp("2026-03-01")).all()
assert pd.to_datetime(counts["last_date"]).le(pd.Timestamp("2026-03-31")).all()


WITH scopes AS (
    SELECT 'March partition' AS scope, * FROM march
    UNION ALL
    SELECT 'GSC-available lane' AS scope, * FROM march
    WHERE gsc_data_available IS TRUE
), per_client AS (
    SELECT scope, client_hash_id, MIN(report_date) AS first_date,
           MAX(report_date) AS last_date, COUNT(DISTINCT report_date) AS observed_dates
    FROM scopes GROUP BY scope, client_hash_id
), spans AS (
    SELECT scope, MIN(first_date) AS earliest_client_start,
           MAX(first_date) AS latest_client_start,
           MIN(last_date) AS earliest_client_end,
           MIN(observed_dates) AS min_dates_per_client,
           MAX(observed_dates) AS max_dates_per_client
    FROM per_client GROUP BY scope
)
SELECT s.scope, COUNT(*) AS rows, MIN(s.report_date) AS first_date,
       MAX(s.report_date) AS last_date, COUNT(DISTINCT s.report_date) AS dates,
       COUNT(DISTINCT s.client_hash_id) AS clients,
       COUNT(DISTINCT (s.client_hash_id, s.content_hash_id)) AS pages,
       ANY

,scope,rows,first_date,last_date,dates,clients,pages,earliest_client_start,latest_client_start,earliest_client_end,min_dates_per_client,max_dates_per_client
0,GSC-available lane,3611061,2026-03-01,2026-03-31,31,47,176738,2026-03-01,2026-03-27,2026-03-15,4,31
1,March partition,9841378,2026-03-01,2026-03-31,31,55,331437,2026-03-01,2026-03-23,2026-03-31,9,31


### Verification query 3 of 3 — availability and missing measurements

The overall row and the FALSE/NULL/TRUE GA4 groups show how many rows survive **`gsc_data_available IS TRUE`**, how many also have GA4, and whether nominally available rows still lack measurements. FALSE and NULL are reported separately, not collapsed into zero activity. GA4 coverage is descriptive context, not an eligibility requirement for this lane.

In [5]:
AVAILABILITY_SQL = """
SELECT CASE WHEN GROUPING(ga4_data_available) = 1 THEN 'ALL'
            WHEN ga4_data_available IS TRUE THEN 'GA4 TRUE'
            WHEN ga4_data_available IS FALSE THEN 'GA4 FALSE'
            ELSE 'GA4 NULL' END AS availability_group,
       COUNT(*) AS source_rows,
       COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_surviving_rows,
       COUNT(*) FILTER (WHERE gsc_data_available IS FALSE) AS gsc_false_rows,
       COUNT(*) FILTER (WHERE gsc_data_available IS NULL) AS gsc_null_rows,
       COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_surviving_rows,
       COUNT(*) FILTER (WHERE gsc_data_available IS TRUE
                         AND ga4_data_available IS TRUE) AS both_surviving_rows,
       COUNT(*) FILTER (WHERE gsc_data_available IS TRUE
                         AND (gsc_impressions IS NULL OR gsc_clicks IS NULL)) AS missing_gsc_counts,
       COUNT(*) FILTER (WHERE gsc_data_available IS TRUE
                         AND (gsc_impressions < 0 OR gsc_clicks < 0)) AS negative_gsc_counts,
       COUNT(*) FILTER (WHERE gsc_data_available IS TRUE
                         AND (gsc_avg_position IS NULL OR gsc_avg_position <= 0)) AS unknown_positions
FROM march
GROUP BY GROUPING SETS ((), (ga4_data_available))
ORDER BY availability_group
"""
print(AVAILABILITY_SQL)
availability = con.sql(AVAILABILITY_SQL).df()
display(availability)
a = availability.loc[availability.availability_group.eq("ALL")].iloc[0]
assert a.gsc_surviving_rows + a.gsc_false_rows + a.gsc_null_rows == a.source_rows
assert a.gsc_surviving_rows == counts.loc[counts.scope.eq("GSC-available lane"), "rows"].iloc[0]
print(f"IS TRUE retains {int(a.gsc_surviving_rows):,}/{int(a.source_rows):,} source rows "
      f"({a.gsc_surviving_rows / a.source_rows:.2%}) for the GSC lane.")


SELECT CASE WHEN GROUPING(ga4_data_available) = 1 THEN 'ALL'
            WHEN ga4_data_available IS TRUE THEN 'GA4 TRUE'
            WHEN ga4_data_available IS FALSE THEN 'GA4 FALSE'
            ELSE 'GA4 NULL' END AS availability_group,
       COUNT(*) AS source_rows,
       COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_surviving_rows,
       COUNT(*) FILTER (WHERE gsc_data_available IS FALSE) AS gsc_false_rows,
       COUNT(*) FILTER (WHERE gsc_data_available IS NULL) AS gsc_null_rows,
       COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_surviving_rows,
       COUNT(*) FILTER (WHERE gsc_data_available IS TRUE
                         AND ga4_data_available IS TRUE) AS both_surviving_rows,
       COUNT(*) FILTER (WHERE gsc_data_available IS TRUE
                         AND (gsc_impressions IS NULL OR gsc_clicks IS NULL)) AS missing_gsc_counts,
       COUNT(*) FILTER (WHERE gsc_data_available IS TRUE
                         AND (gsc_impressions < 0 OR gsc_cli

,availability_group,source_rows,gsc_surviving_rows,gsc_false_rows,gsc_null_rows,ga4_surviving_rows,both_surviving_rows,missing_gsc_counts,negative_gsc_counts,unknown_positions
0,ALL,9841378,3611061,6230317,0,413966,364347,0,0,163189
1,GA4 FALSE,6408671,1718348,4690323,0,0,0,0,0,79833
2,GA4 NULL,3018741,1528366,1490375,0,0,0,0,0,80050
3,GA4 TRUE,413966,364347,49619,0,413966,364347,0,0,3306


IS TRUE retains 3,611,061/9,841,378 source rows (36.69%) for the GSC lane.


### Feature construction — exactly five inputs

This aggregation is the construction step, after the three verification queries above. It returns one row per page with past-window ingredients and separate future-label evidence. It does not return the daily dataset to Python. Coverage requires 14 valid days in each window; all other pages are excluded from the labeled experiment. The feature-frame preview shows five actual rows, with identifiers omitted from the public display.

In [6]:
BUILD_SQL = """
WITH valid_days AS (
    SELECT *, report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-14' AS is_past,
              report_date BETWEEN DATE '2026-03-17' AND DATE '2026-03-30' AS is_future
    FROM march
    WHERE gsc_data_available IS TRUE
      AND gsc_impressions IS NOT NULL AND gsc_impressions >= 0
      AND gsc_clicks IS NOT NULL AND gsc_clicks >= 0
), aggregated AS (
    SELECT client_hash_id, content_hash_id,
           COUNT(*) FILTER (WHERE is_past) AS past_valid_days,
           COUNT(*) FILTER (WHERE is_future) AS future_valid_days,
           MIN(report_date) FILTER (WHERE is_past) AS past_first,
           MAX(report_date) FILTER (WHERE is_past) AS past_last,
           MIN(report_date) FILTER (WHERE is_future) AS future_first,
           MAX(report_date) FILTER (WHERE is_future) AS future_last,
           SUM(gsc_impressions) FILTER (WHERE is_past) AS past_impressions,
           SUM(gsc_clicks) FILTER (WHERE is_past) AS past_clicks,
           SUM(gsc_avg_position * gsc_impressions)
               FILTER (WHERE is_past AND gsc_avg_position > 0) AS weighted_position_sum,
           SUM(gsc_impressions)
               FILTER (WHERE is_past AND gsc_avg_position > 0) AS position_impressions,
           COUNT(*) FILTER (WHERE is_past AND gsc_impressions > 0) AS past_active_days,
           STDDEV_POP(gsc_impressions) FILTER (WHERE is_past) AS past_impression_sd,
           SUM(gsc_impressions) FILTER (WHERE is_future) AS future_impressions
    FROM valid_days GROUP BY client_hash_id, content_hash_id
)
SELECT * FROM aggregated
"""
page_context = con.sql(BUILD_SQL).df()
past_eligible = page_context.past_valid_days.eq(14) & page_context.past_impressions.ge(MIN_PAST_IMPRESSIONS)
labeled_eligible = past_eligible & page_context.future_valid_days.eq(14)
coverage = pd.DataFrame({
    "stage": ["Pages with any valid March GSC day", "Eligible at decision time", "With complete future-label coverage"],
    "pages": [len(page_context), int(past_eligible.sum()), int(labeled_eligible.sum())],
})
display(coverage)
pages = (page_context.loc[labeled_eligible].sort_values(["client_hash_id", "content_hash_id"])
         .copy().reset_index(drop=True))
assert len(pages) >= 2 * K, "Too few eligible pages for the planned experiment."
assert not pages.duplicated(["client_hash_id", "content_hash_id"]).any()
assert pd.to_datetime(pages.past_last).max() < pd.Timestamp("2026-03-17")
assert pd.to_datetime(pages.future_first).min() >= pd.Timestamp("2026-03-17")
pages["decision_date"] = pd.Timestamp("2026-03-17")
X = pd.DataFrame({
    "mean_daily_impressions": pages.past_impressions / 14,
    "ctr_pct": 100 * pages.past_clicks / pages.past_impressions,
    "impression_weighted_position": pages.weighted_position_sum / pages.position_impressions.replace(0, np.nan),
    "active_day_share": pages.past_active_days / 14,
    "daily_impression_cv": pages.past_impression_sd / (pages.past_impressions / 14),
})[FEATURES]
y = (pages.future_impressions < 0.8 * pages.past_impressions).astype("int8").rename("declined")
assert X.shape == (len(pages), 5)
assert not np.isinf(X.to_numpy(dtype=float)).any()
assert y.nunique() == 2, "Both outcome classes are needed; do not report a one-class score."
display(X.head(5))
display(pd.DataFrame({"feature": FEATURES, "missing_rows": X.isna().sum().values,
                      "missing_pct": 100 * X.isna().mean().values}))
display(pd.DataFrame({"declined": [0, 1], "pages": [(y == 0).sum(), (y == 1).sum()]}))
print(f"One modeling row = one client–page at March 17; {len(X):,} rows, exactly five inputs.")

,stage,pages
0,Pages with any valid March GSC day,176738
1,Eligible at decision time,60534
2,With complete future-label coverage,57624


,mean_daily_impressions,ctr_pct,impression_weighted_position,active_day_share,daily_impression_cv
0,8.142857,0.877193,12.421053,1.0,0.321105
1,19.428571,0.000000,12.466912,1.0,0.359618
2,28.500000,0.501253,11.706767,1.0,0.283430
3,112.714286,0.887199,7.308619,1.0,0.703812
4,92.571429,0.000000,56.003858,1.0,0.299108


,feature,missing_rows,missing_pct
0,mean_daily_impressions,0,0.0
1,ctr_pct,0,0.0
2,impression_weighted_position,0,0.0
3,active_day_share,0,0.0
4,daily_impression_cv,0,0.0


,declined,pages
0,0,37039
1,1,20585


One modeling row = one client–page at March 17; 57,624 rows, exactly five inputs.


### The trap — one label-derived column, then remove it

Use one fixed **client holdout** (25%, seed 42), not a page-random split. The same depth-3 decision tree, imputer and held-out clients are used for honest and deliberately leaky fits. **ROC-AUC** is the quick leakage diagnostic: 0.5 is chance ranking, 1.0 is perfect ordering. Also show decline-proxy precision@20 for the queue and its held-out prevalence comparator. Neither score measures editorial usefulness or traffic recovered by refreshing.

Add just `deliberate_label_copy = declined`: it contains the answer, observed only after March 30. A tree can split on it and appear perfect even with client holdout. Then explicitly delete the column and leaky model, refit on the original five features, and keep only the restored honest model/score. Do not tune the model, split, or proxy to the held-out results. This is a teaching experiment, not a model-selection exercise.

In [7]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

def fit_and_score(frame):
    model = make_pipeline(SimpleImputer(strategy="median"),
                          DecisionTreeClassifier(max_depth=3, random_state=SEED))
    model.fit(frame.iloc[train], y.iloc[train])
    probability = model.predict_proba(frame.iloc[test])[:, 1]
    # Deterministic ties retain source aggregation order; they carry no predictive signal.
    top = np.argsort(-probability, kind="stable")[:K]
    return model, probability, {"roc_auc": roc_auc_score(y.iloc[test], probability),
                               "proxy_precision_at_20": float(y.iloc[test].iloc[top].mean())}

# Sort order from a parallel aggregation is not guaranteed; establish deterministic page order first.
order = pages.sort_values(["client_hash_id", "content_hash_id"]).index
pages = pages.loc[order].reset_index(drop=True)
X = X.loc[order].reset_index(drop=True)
y = y.loc[order].reset_index(drop=True)
train, test = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
                   .split(X, y, groups=pages.client_hash_id))
assert set(pages.iloc[train].client_hash_id).isdisjoint(set(pages.iloc[test].client_hash_id))
assert y.iloc[train].nunique() == y.iloc[test].nunique() == 2
assert len(test) >= K and X.iloc[train].notna().any().all()

honest_model, honest_prob, honest_scores = fit_and_score(X)
X_leaky = X.assign(deliberate_label_copy=y)
leaky_model, leaky_prob, leaky_scores = fit_and_score(X_leaky)
assert X_leaky.shape[1] == 6  # Only during this deliberately invalid experiment.
X_leaky.drop(columns=["deliberate_label_copy"], inplace=True)
assert X_leaky.equals(X)
del X_leaky, leaky_model, leaky_prob
model, restored_prob, restored_scores = fit_and_score(X)
assert np.allclose(honest_prob, restored_prob)
assert restored_scores == honest_scores
assert list(X.columns) == FEATURES and model.n_features_in_ == 5
results = pd.DataFrame([honest_scores, leaky_scores, restored_scores],
                       index=["Honest: five past features", "INVALID: added label copy", "Retained: leak deleted"])
display(results)
print(f"Holdout: {pages.iloc[test].client_hash_id.nunique()} clients, {len(test):,} pages; "
      f"decline prevalence {y.iloc[test].mean():.2%}.")
print(f"AUC change after leaking: {leaky_scores['roc_auc'] - honest_scores['roc_auc']:+.4f}.")
print(f"Keep the honest AUC: {restored_scores['roc_auc']:.4f}; leak column/model removed.")
if honest_scores["roc_auc"] >= 0.99:
    print("Honest score is already near-perfect: investigate before claiming the leak caused a jump.")
del honest_model, honest_prob

,roc_auc,proxy_precision_at_20
Honest: five past features,0.603746,0.7
INVALID: added label copy,1.000000,1.0
Retained: leak deleted,0.603746,0.7


Holdout: 8 clients, 43,487 pages; decline prevalence 37.44%.
AUC change after leaking: +0.3963.
Keep the honest AUC: 0.6037; leak column/model removed.


## 4. Data limits

**Named limitation: one month cannot establish seasonal generalization.** Even a client-holdout result in March is not a future-month validation. June remains sealed for later work.

Additional boundaries matter:

- **As-of availability is assumed:** report dates are not ingestion timestamps. The two-day buffer does not prove the values were published, finalized or unrevised on March 17. The features are chronologically separated from the label, but this is not a verified historic production replay.
- **Coverage selection:** requiring future coverage makes the scored sample retrospective and can exclude disappearing pages or clients. The coverage summary quantifies the loss; missing outcomes are never converted to negatives. Complete 14-day windows do not establish a long prior history.
- **Proxy mismatch:** a >20% impression decline can reflect demand, seasonality, SERP changes or technical issues. It is neither a human refresh judgment nor evidence of causal refresh benefit. The Week 2 editorial target remains the eventual success measure.
- **Missingness is patterned:** GSC missingness is audited by GA4 coverage, and position missingness is shown in the feature frame. Five-feature limits prevent adding a position-availability indicator here. Training-only imputation does not eliminate missingness bias.
- **Limited experiment:** one shallow tree, one client split and small top-20 evaluation give a diagnostic, not a stable estimate of deployed performance. We have not optimized on the holdout or read the final month.

**Measured findings:** March has **9,841,378 page-days** across 55 clients. GSC availability
retains **3,611,061 rows (36.69%)** across 47 clients. Past coverage and the exposure floor
retain **60,534 pages**; complete outcome coverage leaves **57,624**, excluding **2,910**
past-eligible pages whose outcomes cannot be fully observed. No feature is missing in this
final frame; that does not imply missingness is absent in the source.

The fixed client holdout contains **8 clients and 43,487 pages**: a quarter of clients can
contain most pages because client sizes differ. The honest **ROC-AUC is 0.6037**, compared
with **1.0000** after deliberately copying the label; deleting that column restores **0.6037**.
Proxy precision@20 is **0.70** versus a held-out decline prevalence of **37.44%**. These are
single-split decline-proxy results, not the independent editorial precision@20 proposed in
Week 2. The modest honest AUC and uneven client representation limit any deployment claim.


In [8]:
print(f"Retrospective coverage loss: {int(past_eligible.sum() - labeled_eligible.sum()):,} "
      "past-eligible pages lack complete future-label coverage.")
print(f"Feature window: {pd.to_datetime(pages.past_first).min().date()} to "
      f"{pd.to_datetime(pages.past_last).max().date()}.")
print(f"Outcome window: {pd.to_datetime(pages.future_first).min().date()} to "
      f"{pd.to_datetime(pages.future_last).max().date()}.")
assert len(FEATURES) == X.shape[1] == model.n_features_in_ == 5
assert "deliberate_label_copy" not in X.columns
assert "leaky_model" not in globals() and "X_leaky" not in globals()
print("Final state: five past-window features, honest model, no label-derived predictor.")

Retrospective coverage loss: 2,910 past-eligible pages lack complete future-label coverage.
Feature window: 2026-03-01 to 2026-03-14.
Outcome window: 2026-03-17 to 2026-03-30.
Final state: five past-window features, honest model, no label-derived predictor.


## 5. Self-check

- [x] Five plain-words contract answers, with separate source and model grains.
- [x] Exactly three verification queries written; availability uses `IS TRUE`.
- [x] Exactly five candidate features, each with an available-when explanation.
- [x] Deliberate one-column leak and explicit deletion implemented on the same client split.
- [x] Named slice limitations and distinguished decline proxy from refresh benefit.
- [x] All cells executed successfully against the gated March warehouse partition; outputs inspected.
- [x] Honest and leaky scores measured, leak removed, and final five-feature state verified.
**Submission:** Commit and push this executed notebook at `work/notebooks/w03_data_contract.ipynb`, then submit the repository URL on the card.

**AI assistance:** An assistant prepared and executed this contract and code on the real March warehouse partition using the requested repo skills.
The intern should review the window, exposure floor and proxy choices. All reported query results and scores come from the saved execution. No raw dataset is committed. Portal submission
is a separate step.